# Public-Only Teacher-Trace SFT

Conservative QLoRA SFT on Claude-generated five-section traces for the public train split only. This notebook prepares chat JSONL for `train_qlora_sft.py`, trains from the base Qwen model, and evaluates the adapter on the held-out public split.

## 1. Mount Drive and Set Paths

Set `REPO_ROOT` if your repo is not in one of the default locations below. The expected raw trace file is `data_public_teacher/public_teacher_traces.jsonl` under the repo root.

In [ ]:
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

candidate_roots = []
if os.environ.get('REPO_ROOT'):
    candidate_roots.append(Path(os.environ['REPO_ROOT']).expanduser())
candidate_roots.extend([
    Path.cwd(),
    Path('/content/151B_SP26_Competition'),
    Path('/content/drive/MyDrive/qwen_math_comp/151B_SP26_Competition'),
])

REPO_ROOT = None
for root in candidate_roots:
    if (root / 'train_qlora_sft.py').exists():
        REPO_ROOT = root.resolve()
        break

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not find repo root containing train_qlora_sft.py. '
        'Set os.environ["REPO_ROOT"] to the 151B_SP26_Competition path and rerun this cell.'
    )

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ['REPO_ROOT'] = str(REPO_ROOT)

DATA_DIR = Path('data_public_teacher')
RAW_TRACE_PATH = DATA_DIR / 'public_teacher_traces.jsonl'
PUBLIC_TRAIN_SPLIT = Path('data/train.jsonl')
PUBLIC_HELDOUT_SPLIT = Path('data/test.jsonl')
SFT_TRAIN_PATH = DATA_DIR / 'public_train_sft.jsonl'
SFT_LOSS_VAL_PATH = DATA_DIR / 'public_loss_val_sft.jsonl'
MANIFEST_PATH = DATA_DIR / 'manifest.json'
OUTPUT_DIR = Path('outputs/qwen3_4b_public_teacher_cot_structured_r8_lr2e5')
SMOKE_OUTPUT_DIR = Path('outputs/debug_public_teacher_traces_5steps')

MODEL_ID = 'Qwen/Qwen3-4B-Thinking-2507'
SEED = 151
LOSS_VAL_FRACTION = 0.10
PRIMARY_MAX_SEQ_LEN = 8192

print('IN_COLAB =', IN_COLAB)
print('REPO_ROOT =', REPO_ROOT)
print('RAW_TRACE_PATH =', RAW_TRACE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 2. Install Dependencies

This installs the repo package and its notebook extra into the active Colab runtime.

In [ ]:
%pip install -q -e ".[notebook]"

## 3. Verify GPU

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
    free, total = torch.cuda.mem_get_info()
    print(f'gpu memory free/total: {free / 2**30:.2f} GiB / {total / 2**30:.2f} GiB')
else:
    raise RuntimeError('A CUDA GPU is required for this QLoRA notebook.')


## 4. Load and Validate Teacher Traces

The filter is intentionally strict: it keeps only train-split traces with correct gold answers and the exact final `Answer: \boxed{...}` format.

In [ ]:
import json
import math
import random
import re
from collections import Counter
from pathlib import Path

from shared.scoring import score_one

SECTION_LABELS = ('Parse:', 'Plan:', 'Compute:', 'Verify:', 'Answer:')
FINAL_ANSWER_RE = re.compile(r'^Answer:\s*\\boxed\{(.+)\}\s*$')
UNCERTAINTY_RE = re.compile(
    r"\b(maybe|i think|i am not sure|i'm not sure|not sure|uncertain|probably|might be)\b",
    flags=re.IGNORECASE,
)

SFT_SYSTEM = """You are generating supervised fine-tuning data for a small math reasoning model.

Solve the math problem in exactly five sections:
Parse: restate what is being asked in one sentence.
Plan: give the main strategy in one sentence.
Compute: show the necessary math steps clearly. This is the only section where detailed work should appear.
Verify: give one short sanity check, consistency check, or reason the final answer fits the problem.
Answer: put all final answers inside one \\boxed{}.

For multiple-choice, the Answer line must contain only the option letter inside \\boxed{}.
Do not write anything after the final boxed answer."""


def load_jsonl(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def final_nonempty_line(text: str) -> str:
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return lines[-1] if lines else ''


def boxed_contents(text: str) -> list[str]:
    contents = []
    marker = r'\boxed'
    i = 0
    while True:
        start = text.find(marker, i)
        if start < 0:
            break
        brace = text.find('{', start + len(marker))
        if brace < 0:
            i = start + len(marker)
            continue
        depth = 0
        for j in range(brace, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    contents.append(text[brace + 1:j].strip())
                    i = j + 1
                    break
        else:
            i = brace + 1
    return contents


def format_user_turn(item: dict) -> str:
    question = str(item['question']).strip()
    options = item.get('options')
    if not options:
        return question
    labels = [chr(65 + i) for i in range(len(options))]
    options_text = '\n'.join(f'{label}. {str(option).strip()}' for label, option in zip(labels, options))
    return f'{question}\n\nOptions:\n{options_text}'


def validate_teacher_trace(item: dict, trace: str) -> tuple[list[str], str]:
    reasons = []
    trace = trace.strip()
    if not trace:
        return ['empty_teacher_trace'], ''

    missing_sections = [label for label in SECTION_LABELS if label not in trace]
    if missing_sections:
        reasons.append('missing_sections')

    final_line = final_nonempty_line(trace)
    final_match = FINAL_ANSWER_RE.match(final_line)
    if not final_match:
        reasons.append('bad_final_answer_line')

    boxes = boxed_contents(trace)
    if not boxes:
        reasons.append('no_boxed_answer')
    elif len(set(boxes)) > 1:
        reasons.append('conflicting_boxed_answers')

    if item.get('options') and final_match:
        boxed = final_match.group(1).strip()
        if not re.fullmatch(r'[A-Za-z]', boxed):
            reasons.append('mcq_box_not_single_letter')

    if UNCERTAINTY_RE.search(trace):
        reasons.append('uncertainty_marker')

    scored = score_one(item, trace)
    if not scored.correct:
        reasons.append('wrong_answer')

    return reasons, scored.extracted


for required_path in [RAW_TRACE_PATH, PUBLIC_TRAIN_SPLIT, PUBLIC_HELDOUT_SPLIT]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

raw_trace_rows = load_jsonl(RAW_TRACE_PATH)
train_split_rows = load_jsonl(PUBLIC_TRAIN_SPLIT)
heldout_rows = load_jsonl(PUBLIC_HELDOUT_SPLIT)

trace_by_id = {}
duplicate_ids = []
for row in raw_trace_rows:
    if 'id' not in row:
        continue
    public_id = int(row['id'])
    if public_id in trace_by_id:
        duplicate_ids.append(public_id)
    trace_by_id[public_id] = row

if duplicate_ids:
    raise ValueError(f'Duplicate ids in {RAW_TRACE_PATH}: {sorted(set(duplicate_ids))[:20]}')

valid_records = []
dropped_rows = []
drop_reason_counts = Counter()

for item in train_split_rows:
    public_id = int(item['id'])
    raw = trace_by_id.get(public_id)
    if raw is None:
        reasons = ['missing_teacher_trace_row']
        dropped_rows.append({'id': public_id, 'reasons': reasons})
        drop_reason_counts.update(reasons)
        continue

    trace = str(raw.get('teacher_trace', '')).strip()
    reasons, extracted = validate_teacher_trace(item, trace)
    if reasons:
        dropped_rows.append({'id': public_id, 'reasons': reasons})
        drop_reason_counts.update(set(reasons))
        continue

    valid_records.append({
        'public_id': public_id,
        'item': item,
        'teacher_trace': trace,
        'extracted': extracted,
    })

print(f'raw trace rows: {len(raw_trace_rows)}')
print(f'public train split rows: {len(train_split_rows)}')
print(f'held-out public rows reserved for final eval: {len(heldout_rows)}')
print(f'valid train traces: {len(valid_records)}')
print(f'dropped train traces: {len(dropped_rows)}')
print('drop reasons:')
for reason, count in drop_reason_counts.most_common():
    print(f'  {reason}: {count}')

if len(valid_records) < 20:
    raise RuntimeError('Too few valid teacher traces after filtering; inspect drop reasons before training.')


## 5. Token Lengths and Dataset Stats

In [ ]:
from statistics import mean

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def make_messages(item: dict, teacher_trace: str) -> list[dict]:
    return [
        {'role': 'system', 'content': SFT_SYSTEM},
        {'role': 'user', 'content': format_user_turn(item)},
        {'role': 'assistant', 'content': teacher_trace},
    ]


def chat_token_len(messages: list[dict]) -> int:
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return len(tokenizer(rendered, add_special_tokens=False)['input_ids'])


def percentile(values: list[int], q: float) -> float:
    if not values:
        return float('nan')
    ordered = sorted(values)
    pos = (len(ordered) - 1) * q
    lo = math.floor(pos)
    hi = math.ceil(pos)
    if lo == hi:
        return float(ordered[lo])
    return float(ordered[lo] + (ordered[hi] - ordered[lo]) * (pos - lo))

examples = []
for record in valid_records:
    messages = make_messages(record['item'], record['teacher_trace'])
    n_tokens = chat_token_len(messages)
    examples.append({
        'messages': messages,
        'source': 'public_jsonl_claude_teacher',
        'kind': 'mcq' if record['item'].get('options') else 'free_response',
        'public_id': record['public_id'],
        'answer': record['item'].get('answer'),
        'teacher_model': 'claude-opus-4.7',
        'n_tokens': n_tokens,
        'extracted': record['extracted'],
    })

tokens = [ex['n_tokens'] for ex in examples]
mcq_count = sum(1 for ex in examples if ex['kind'] == 'mcq')
free_count = len(examples) - mcq_count
too_long_8192 = sum(1 for t in tokens if t > 8192)
too_long_16384 = sum(1 for t in tokens if t > 16384)

print(f'valid examples: {len(examples)}')
print(f'MCQ: {mcq_count} ({mcq_count / len(examples):.1%})')
print(f'free-response: {free_count} ({free_count / len(examples):.1%})')
print(f'token avg: {mean(tokens):.1f}')
print(f'token p50: {percentile(tokens, 0.50):.0f}')
print(f'token p90: {percentile(tokens, 0.90):.0f}')
print(f'token p95: {percentile(tokens, 0.95):.0f}')
print(f'token max: {max(tokens)}')
print(f'examples over 8192 tokens: {too_long_8192}')
print(f'examples over 16384 tokens: {too_long_16384}')

if mean(tokens) < 600:
    print('WARNING: average token length is very low; inspect traces for over-compression before training.')
if too_long_8192:
    print('NOTE: train_qlora_sft.py will skip rows over --max_seq_len=8192. Consider the optional 16k run if many long traces are high quality.')

print('\nSample SFT row:')
print(json.dumps(examples[0], indent=2, ensure_ascii=False)[:4000])


## 6. Write Train and Loss-Validation SFT JSONL

The held-out `data/test.jsonl` rows are not used here. The 10% split below is only for Trainer eval loss during SFT.

In [ ]:
rng = random.Random(SEED)
shuffled_examples = list(examples)
rng.shuffle(shuffled_examples)

loss_val_n = max(1, round(len(shuffled_examples) * LOSS_VAL_FRACTION))
if len(shuffled_examples) - loss_val_n < 1:
    raise RuntimeError('Not enough examples to create both train and loss-val splits.')

loss_val_examples = shuffled_examples[:loss_val_n]
train_examples = shuffled_examples[loss_val_n:]

write_jsonl(SFT_TRAIN_PATH, train_examples)
write_jsonl(SFT_LOSS_VAL_PATH, loss_val_examples)

manifest = {
    'raw_trace_path': str(RAW_TRACE_PATH),
    'public_train_split': str(PUBLIC_TRAIN_SPLIT),
    'public_heldout_split': str(PUBLIC_HELDOUT_SPLIT),
    'sft_train_path': str(SFT_TRAIN_PATH),
    'sft_loss_val_path': str(SFT_LOSS_VAL_PATH),
    'seed': SEED,
    'loss_val_fraction': LOSS_VAL_FRACTION,
    'raw_trace_rows': len(raw_trace_rows),
    'public_train_rows': len(train_split_rows),
    'heldout_rows_reserved_for_final_eval': len(heldout_rows),
    'valid_examples': len(examples),
    'train_examples': len(train_examples),
    'loss_val_examples': len(loss_val_examples),
    'dropped_rows': len(dropped_rows),
    'drop_reason_counts': dict(drop_reason_counts),
    'mcq_examples': mcq_count,
    'free_response_examples': free_count,
    'token_stats': {
        'avg': mean(tokens),
        'p50': percentile(tokens, 0.50),
        'p90': percentile(tokens, 0.90),
        'p95': percentile(tokens, 0.95),
        'max': max(tokens),
        'over_8192': too_long_8192,
        'over_16384': too_long_16384,
    },
}
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'wrote {len(train_examples)} train rows -> {SFT_TRAIN_PATH}')
print(f'wrote {len(loss_val_examples)} loss-val rows -> {SFT_LOSS_VAL_PATH}')
print(f'wrote manifest -> {MANIFEST_PATH}')


## 7. Optional 5-Step Smoke Training

Run this before the full training cell. It verifies model load, QLoRA setup, tokenization, and adapter save.

In [ ]:
import subprocess

subprocess.run([
    'python', 'train_qlora_sft.py',
    '--train_path', str(SFT_TRAIN_PATH),
    '--eval_path', str(SFT_LOSS_VAL_PATH),
    '--output_dir', str(SMOKE_OUTPUT_DIR),
    '--max_seq_len', '8192',
    '--max_steps', '5',
    '--per_device_train_batch_size', '1',
    '--per_device_eval_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--learning_rate', '2e-5',
    '--lora_r', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--eval_steps', '5',
    '--save_steps', '5',
    '--logging_steps', '1',
], check=True)


## 8. Full Conservative QLoRA Training

In [ ]:
import subprocess

subprocess.run([
    'python', 'train_qlora_sft.py',
    '--train_path', str(SFT_TRAIN_PATH),
    '--eval_path', str(SFT_LOSS_VAL_PATH),
    '--output_dir', str(OUTPUT_DIR),
    '--max_seq_len', '8192',
    '--num_train_epochs', '1',
    '--per_device_train_batch_size', '1',
    '--per_device_eval_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--learning_rate', '2e-5',
    '--lora_r', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--eval_steps', '25',
    '--save_steps', '25',
    '--logging_steps', '5',
], check=True)


## 9. Optional 16k / Lower-LR Ablation

Use this only if the token histogram shows many high-quality traces above 8192 tokens and the GPU has enough memory.

In [ ]:
RUN_16K_ABLATION = False

if RUN_16K_ABLATION:
    import subprocess
    output_dir_16k = Path('outputs/qwen3_4b_public_teacher_cot_structured_r8_lr1e5_16k')
    cmd = [
        'python', 'train_qlora_sft.py',
        '--train_path', str(SFT_TRAIN_PATH),
        '--eval_path', str(SFT_LOSS_VAL_PATH),
        '--output_dir', str(output_dir_16k),
        '--max_seq_len', '16384',
        '--num_train_epochs', '1',
        '--per_device_train_batch_size', '1',
        '--per_device_eval_batch_size', '1',
        '--gradient_accumulation_steps', '16',
        '--learning_rate', '1e-5',
        '--lora_r', '8',
        '--lora_alpha', '16',
        '--lora_dropout', '0.05',
        '--eval_steps', '25',
        '--save_steps', '25',
        '--logging_steps', '5',
    ]
    subprocess.run(cmd, check=True)
else:
    print('Set RUN_16K_ABLATION = True and rerun this cell to train the optional 16k variant.')


## 10. Evaluate Adapter on Held-Out Public Split

This uses `eval=full` together with `eval.data_path=data/test.jsonl`, so the full slice is the 226-row held-out split, not the whole public file.

In [ ]:
import subprocess

ADAPTER_PATH = OUTPUT_DIR / 'final_adapter'
if not (ADAPTER_PATH / 'adapter_config.json').exists():
    raise FileNotFoundError(f'Missing trained adapter under {ADAPTER_PATH}')

subprocess.run([
    'python', '-m', 'experiments.prompt_engineering.src.eval',
    'run=cot_structured_only',
    'eval=full',
    f'eval.data_path={PUBLIC_HELDOUT_SPLIT}',
    'eval.max_tokens=16384',
    'regime=greedy',
    f'runner.adapter_path={ADAPTER_PATH}',
    'run_name=sft_public_teacher_cot_structured_heldout_16384',
], check=True)


## 11. Base Baseline on the Same Held-Out Split

Run this once if you do not already have the matching base result. Compare the leaderboard metrics against the adapter run above.

In [ ]:
import subprocess

subprocess.run([
    'python', '-m', 'experiments.prompt_engineering.src.eval',
    'run=cot_structured_only',
    'eval=full',
    f'eval.data_path={PUBLIC_HELDOUT_SPLIT}',
    'eval.max_tokens=16384',
    'regime=greedy',
    'run_name=base_cot_structured_heldout_16384',
], check=True)


## 12. Optional Full Public Sanity Check

This is not the primary held-out metric because it includes the 900 train-split rows. Use it only to compare with older full-public experiment logs.

In [ ]:
RUN_FULL_PUBLIC_SANITY = False

if RUN_FULL_PUBLIC_SANITY:
    import subprocess
    subprocess.run([
        'python', '-m', 'experiments.prompt_engineering.src.eval',
        'run=cot_structured_only',
        'eval=full',
        'eval.max_tokens=16384',
        'regime=greedy',
        f'runner.adapter_path={ADAPTER_PATH}',
        'run_name=sft_public_teacher_cot_structured_full_public_16384',
    ], check=True)
else:
    print('Set RUN_FULL_PUBLIC_SANITY = True to evaluate on all 1,126 public rows.')
